In [1]:
import torch
import torch.nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from pathlib import Path
import matplotlib.pyplot as plt
from pytorch3d.loss import chamfer_distance
import numpy as np
np.bool8 = np.bool
from torch.utils.tensorboard.writer import SummaryWriter
import datetime
from decoder import *
from utils import *
from pytorch3d.loss import chamfer_distance
import open3d as o3d
from preprocessing import *
from dataset import *
import train
import trimesh

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
num_points = 2048
empty_the_data_folder = True

In [4]:
root = Path("../../../GraspDataset")
print(f"Project Root: {root}")

# delete the previous files in the output folder
if empty_the_data_folder:
    folder_path = Path("../../../GraspDataset/data/preprocessed/preprocessing3")
    # Iterate through all items in the folder
    for file in folder_path.iterdir():
        if file.is_file():
            file.unlink()

mesh_path_list = get_random_shapenet_meshes(seed=42, num_objects=100, root=root)
for path in mesh_path_list:
    print(path)
    
# path_str = "/home/nikola/Projects/tum-adlr-ss26-07/data/studentGrasping/student_grasps_v1/02876657/9fe7e6a7bf8ca964efad53eb3f0b36fa/3/mesh.obj"
# mesh_path_list = [Path(path_str)]

Project Root: ..\..\..\GraspDataset
..\..\..\GraspDataset\data\studentGrasping\student_grasps_v1\03636649\77a7d38645738e2212c5719ce6179\0\mesh.obj
..\..\..\GraspDataset\data\studentGrasping\student_grasps_v1\02876657\4d4fc73864844dad1ceb7b8cc3792fd\8\mesh.obj
..\..\..\GraspDataset\data\studentGrasping\student_grasps_v1\02808440\bdacdeb89e174d743321831d2245cf06\0\mesh.obj
..\..\..\GraspDataset\data\studentGrasping\student_grasps_v1\03797390\67b9abb424cf22a22d7082a28b056a5\9\mesh.obj
..\..\..\GraspDataset\data\studentGrasping\student_grasps_v1\02876657\cc48fe97a95e8716ccaa5ad584801c3e\2\mesh.obj
..\..\..\GraspDataset\data\studentGrasping\student_grasps_v1\02876657\b2498accc1c3fe732db3066d0100ee4\7\mesh.obj
..\..\..\GraspDataset\data\studentGrasping\student_grasps_v1\02876657\a1275bd03ab15100f6dbe3dc17d6cdf7\2\mesh.obj
..\..\..\GraspDataset\data\studentGrasping\student_grasps_v1\02876657\63f60daa8d254f445200bdb78a83bb9b\0\mesh.obj
..\..\..\GraspDataset\data\studentGrasping\student_grasps_

In [14]:
mesh_list = []
for path in mesh_path_list:
    mesh_list.append(trimesh.load(path, force='mesh', skip_materials=True))

for index, mesh in enumerate(mesh_list):
    output_path = root / Path("data/preprocessed/preprocessing3") / f"mesh{index}.obj"
    mesh.export(output_path)  # Save the cleaned mesh for inspection

# Sample points from the surface
# returns a (point_num, 3) NumPy array
point_cloud_list = []
for mesh in mesh_list:
    points = mesh.sample(num_points)
    # Create a PointCloud object for Trimesh utilities
    point_cloud_list.append(trimesh.points.PointCloud(points))


for point_cloud in point_cloud_list:
    point_cloud.vertices = normalize_statistically_points(point_cloud.vertices)


for index, point_cloud in enumerate(point_cloud_list):
    # Ensure output directory exists and save
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path = root / Path("data/preprocessed/preprocessing3") / f"pointcloud{index}.obj"
    point_cloud.export(output_path)
    print(f"Saved preprocessed point cloud to: {output_path}")

Saved preprocessed point cloud to: ..\..\..\GraspDataset\data\preprocessed\preprocessing3\pointcloud0.obj
Saved preprocessed point cloud to: ..\..\..\GraspDataset\data\preprocessed\preprocessing3\pointcloud1.obj
Saved preprocessed point cloud to: ..\..\..\GraspDataset\data\preprocessed\preprocessing3\pointcloud2.obj
Saved preprocessed point cloud to: ..\..\..\GraspDataset\data\preprocessed\preprocessing3\pointcloud3.obj
Saved preprocessed point cloud to: ..\..\..\GraspDataset\data\preprocessed\preprocessing3\pointcloud4.obj
Saved preprocessed point cloud to: ..\..\..\GraspDataset\data\preprocessed\preprocessing3\pointcloud5.obj
Saved preprocessed point cloud to: ..\..\..\GraspDataset\data\preprocessed\preprocessing3\pointcloud6.obj
Saved preprocessed point cloud to: ..\..\..\GraspDataset\data\preprocessed\preprocessing3\pointcloud7.obj
Saved preprocessed point cloud to: ..\..\..\GraspDataset\data\preprocessed\preprocessing3\pointcloud8.obj
Saved preprocessed point cloud to: ..\..\..\Gr

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


HyperParameters = {
    "batch_size": 9,
    "learning_rate": 2e-4,
    "num_epochs": 50000,
    "hidden_dim": 64,
    "latent_dim": 32,
    "num_timesteps": 1000,
    "num_points": 256,
    "root": Path("../../../GraspDataset/data/preprocessed/preprocessing3")
}

dataset = PointCloudDataset(HyperParameters["root"], number_points=HyperParameters["num_points"], max_objects=9)
dataset2 = PointCloudDataset(HyperParameters["root"], number_points=2048, max_objects=9)

dataloader = DataLoader(dataset, shuffle=False, batch_size=HyperParameters["batch_size"])
epochs = HyperParameters["num_epochs"]

autoencoder = AutoEncoder(HyperParameters["num_points"], 3, HyperParameters["hidden_dim"],
                           HyperParameters["latent_dim"], HyperParameters["num_timesteps"])

num_parameters = sum(p.numel() for p in autoencoder.parameters() if p.requires_grad)
print(f"Trainable Parameters: {num_parameters / 1e6:.2f}M")



optimizer = torch.optim.Adam(autoencoder.parameters(), lr=HyperParameters["learning_rate"])
#scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=T_max // 4, gamma=0.5)
scheduler = None
log_path = Path(f"logs/training/{datetime.datetime.now().strftime('%b%d_%H_%M')}_dpm_autoencoder_4.pt")
writer = SummaryWriter(log_path)

Trainable Parameters: 0.63M


In [7]:
print(len(dataset))

0


In [4]:
ae_loss_history = []
kld_loss_history = []

for epoch in range(2):
    best_loss = 1e6
    ae_loss, kld_loss = train.train_ae(dataloader, autoencoder, optimizer, writer)

    
    ae_loss_history.append(ae_loss)
    kld_loss_history.append(kld_loss)

    if scheduler is not None:
        scheduler.step()

    if ae_loss < best_loss:
        best_loss = ae_loss
        torch.save(autoencoder.state_dict(), Path.cwd() / Path("models/dpm_autoencoder_best4.pt"))

    if epoch % 5000 == 0:
        print(f"Current Epoch Training Loss: {ae_loss:.6f}")

torch.save(autoencoder.state_dict(), Path.cwd() / Path("models/dpm_decoder_end_4.pt"))

fig = plt.figure()

plt.title("AE Loss over Epoch")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.plot(ae_loss_history, color="b")
plt.plot()

fig = plt.figure()

plt.title("KLD Loss over Epoch")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.plot(kld_loss_history, color="m")
plt.plot()
plt.show()

NameError: name 'np' is not defined